# Final Dataset Merger
Combines **Mortality**, **VegData** (tree cover loss), and **AQI** into one long-format dataset.

**Join key**: `state + county + year`  
**Output**: one row per county per year (2000–2016)

In [ ]:
import pandas as pd
from functools import reduce
from pathlib import Path

# ── Configuration — update these paths ───────────────────────────────────────
DATA_DIR     = Path(".")                     # folder with Mortality_YYYY.csv files
AQI_PATH     = Path("combined_aqi.csv")      # your AQI file
VEG_URL      = (
    "https://docs.google.com/spreadsheets/d/e/"
    "2PACX-1vTU6WAz3UfcApC4kZVCDhdYwU8FemgPqRv4DKM9QqAwoeD3xZbCSBRxCqgYlunKUw"
    "/pub?gid=787104991&single=true&output=csv"
)

YEARS        = range(2000, 2017)             # 2000 through 2016 inclusive
FILE_PATTERN = "Mortality_{year}.csv"
VEG_THRESHOLD = 0                            # use threshold=0 (all tree cover)

print("Configuration set.")

## Helper: county name normalizer
Each dataset stores county names differently:
- Mortality: `"Autauga County, AL"`
- VegData:   `subnational2 = "Autauga"`, `subnational1 = "Alabama"`
- AQI:       `County Name = "Autauga"`, `State Name = "Alabama"`

We normalise everything to lowercase bare county name + full state name.

In [ ]:
# US state abbreviation → full name lookup
STATE_ABBR = {
    'AL':'Alabama','AK':'Alaska','AZ':'Arizona','AR':'Arkansas','CA':'California',
    'CO':'Colorado','CT':'Connecticut','DE':'Delaware','FL':'Florida','GA':'Georgia',
    'HI':'Hawaii','ID':'Idaho','IL':'Illinois','IN':'Indiana','IA':'Iowa',
    'KS':'Kansas','KY':'Kentucky','LA':'Louisiana','ME':'Maine','MD':'Maryland',
    'MA':'Massachusetts','MI':'Michigan','MN':'Minnesota','MS':'Mississippi',
    'MO':'Missouri','MT':'Montana','NE':'Nebraska','NV':'Nevada','NH':'New Hampshire',
    'NJ':'New Jersey','NM':'New Mexico','NY':'New York','NC':'North Carolina',
    'ND':'North Dakota','OH':'Ohio','OK':'Oklahoma','OR':'Oregon','PA':'Pennsylvania',
    'RI':'Rhode Island','SC':'South Carolina','SD':'South Dakota','TN':'Tennessee',
    'TX':'Texas','UT':'Utah','VT':'Vermont','VA':'Virginia','WA':'Washington',
    'WV':'West Virginia','WI':'Wisconsin','WY':'Wyoming','DC':'District of Columbia',
}

def normalize_county(raw: str) -> str:
    """'Autauga County, AL'  →  'autauga'"""
    name = raw.split(",")[0]                # drop ', AL'
    for suffix in [" County", " Parish", " Borough",
                   " Census Area", " Municipality", " City"]:
        name = name.replace(suffix, "")
    return name.strip().lower()

def normalize_state_from_mortality(raw: str) -> str:
    """'Autauga County, AL'  →  'alabama'"""
    abbr = raw.split(",")[-1].strip()
    return STATE_ABBR.get(abbr, abbr).lower()

print("Helper functions defined.")

## 1 — Load & reshape Mortality → long format

In [ ]:
mort_dfs = []
missing  = []

for year in YEARS:
    fp = DATA_DIR / FILE_PATTERN.format(year=year)
    if not fp.exists():
        print(f"  WARNING: {fp.name} not found — skipping")
        missing.append(year)
        continue

    temp = pd.read_csv(
        fp,
        usecols=["County", "County Code", "Crude Rate"],
        dtype={"County": "str", "County Code": "str", "Crude Rate": "str"},
    )

    # Drop summary/footer rows
    temp = temp[temp["County Code"].notna() & (temp["County Code"].str.strip() != "")]

    # Clean crude rate — strips '(Unreliable)' and other text
    temp["crude_rate"] = (
        temp["Crude Rate"]
        .str.replace(r"[^\d.]", "", regex=True)
        .replace("", float("nan"))
        .astype(float)
    )

    # Normalise join keys
    temp["county"] = temp["County"].apply(normalize_county)
    temp["state"]  = temp["County"].apply(normalize_state_from_mortality)
    temp["year"]   = year
    temp["fips"]   = temp["County Code"].str.strip()

    mort_dfs.append(temp[["fips", "state", "county", "year", "crude_rate"]])

mortality_long = pd.concat(mort_dfs, ignore_index=True)

if missing:
    print(f"Missing years: {missing}")
print(f"Mortality long shape: {mortality_long.shape}")
mortality_long.head()

## 2 — Load & reshape VegData → long format

In [ ]:
veg_raw = pd.read_csv(VEG_URL)

# Keep only threshold=0 (all tree cover)
veg = veg_raw[veg_raw["threshold"] == VEG_THRESHOLD].copy()

# Normalise join keys
veg["state"]  = veg["subnational1"].str.strip().str.lower()
veg["county"] = veg["subnational2"].str.strip().str.lower()

# Select area/extent columns + yearly loss columns for 2000-2016
static_cols = ["state", "county", "area_ha", "extent_2000_ha", "extent_2010_ha", "gain_2000-2020_ha"]
loss_cols   = [c for c in veg.columns if c.startswith("tc_loss_ha_")]

# Filter loss cols to years 2001-2016 only (loss starts at 2001)
loss_cols_in_range = [c for c in loss_cols if int(c.split("_")[-1]) in range(2001, 2017)]

veg_subset = veg[static_cols + loss_cols_in_range]

# Melt to long format
veg_long = veg_subset.melt(
    id_vars=static_cols,
    value_vars=loss_cols_in_range,
    var_name="year",
    value_name="tc_loss_ha",
)
veg_long["year"] = veg_long["year"].str.extract(r"(\d{4})").astype(int)

print(f"VegData long shape: {veg_long.shape}")
veg_long.head()

## 3 — Load & aggregate AQI → one row per county per year

In [ ]:
aqi_raw = pd.read_csv(AQI_PATH)

# Normalise join keys
aqi_raw["state"]  = aqi_raw["State Name"].str.strip().str.lower()
aqi_raw["county"] = aqi_raw["County Name"].str.strip().str.lower()
aqi_raw["year"]   = aqi_raw["Year"].astype(int)

# Filter to 2000–2016
aqi_raw = aqi_raw[aqi_raw["year"].between(2000, 2016)]

# Aggregate: mean of Arithmetic Mean across all parameters & sites per county-year
aqi_long = (
    aqi_raw
    .groupby(["state", "county", "year"], as_index=False)
    .agg(
        aqi_mean=("Arithmetic Mean", "mean"),
        aqi_std =("Arithmetic Standard Dev", "mean"),
        aqi_max =("1st Max Value", "max"),
        n_aqi_records=("Arithmetic Mean", "count"),
    )
)

print(f"AQI long shape: {aqi_long.shape}")
aqi_long.head()

## 4 — Merge all three datasets

In [ ]:
# Start with mortality as the base (it defines the county-year universe)
final = mortality_long.merge(
    veg_long.drop(columns=["area_ha", "extent_2000_ha", "extent_2010_ha", "gain_2000-2020_ha"]),
    on=["state", "county", "year"],
    how="left",
)

final = final.merge(
    aqi_long,
    on=["state", "county", "year"],
    how="left",
)

# Also bring in static veg columns (same for all years per county)
veg_static = veg_long[["state", "county", "area_ha", "extent_2000_ha",
                        "extent_2010_ha", "gain_2000-2020_ha"]].drop_duplicates()
final = final.merge(veg_static, on=["state", "county"], how="left")

# Sort for readability
final = final.sort_values(["state", "county", "year"]).reset_index(drop=True)

print(f"Final dataset shape: {final.shape}")
print(f"Columns: {final.columns.tolist()}")
final.head(20)

## 5 — Check merge coverage

In [ ]:
total = len(final)
veg_matched  = final["tc_loss_ha"].notna().sum()
aqi_matched  = final["aqi_mean"].notna().sum()

print(f"Total rows          : {total:,}")
print(f"Rows with VegData   : {veg_matched:,}  ({veg_matched/total:.1%})")
print(f"Rows with AQI data  : {aqi_matched:,}  ({aqi_matched/total:.1%})")
print()

# Counties that didn't match VegData — likely name spelling mismatches
unmatched_veg = final[final["tc_loss_ha"].isna()][["state","county"]].drop_duplicates()
print(f"Counties without VegData match: {len(unmatched_veg)}")
if len(unmatched_veg) > 0:
    print(unmatched_veg.head(20).to_string(index=False))

In [ ]:
# ── Optional: inspect & fix name mismatches ───────────────────────────────────
# If counties didn't match, check what VegData calls them:
# veg_long[veg_long["state"] == "louisiana"]["county"].unique()  # e.g. parishes
#
# Then add manual overrides:
# COUNTY_OVERRIDES = {
#     ("louisiana", "st. tammany")  : "saint tammany",
#     ("alaska",    "wade hampton") : "kusilvak",
# }
# Apply before the merge step above if needed.
print("(No overrides needed — uncomment above if you see mismatches)")

## 6 — Save final dataset

In [ ]:
out_path = DATA_DIR / "final_dataset_2000_2016.csv"
final.to_csv(out_path, index=False)
print(f"Saved to: {out_path.resolve()}")
print(f"Shape: {final.shape[0]:,} rows × {final.shape[1]} columns")
print(f"\nColumn summary:")
print(final.dtypes.to_string())